# Cellina AE / GCN vs camera-ready

New runs (JSONs in `DATA_ROOT/datasets/crc/correlations`) compared against the camera-ready CSV on all CRC slides. Mean $\pm$ std over all (slide, held-out cell type) pairs, as in `make_table.ipynb`.

- base: `cellina` (csv) vs `cellina-ae-cf` (json)
- graph: `cellina-gat` (csv) vs `cellina-gcn-cf` (json)

In [1]:
import os
import json
import numpy as np
import pandas as pd

pd.set_option('display.width', 200)

In [2]:
dataset_name = "crc"
DATA_ROOT = os.environ.get("DATA_ROOT", ".")  # JSON_BASE_PATH
SUMMARY_CSV = f'../results/loo_summary_{dataset_name}_DEG_50_camready.csv'
OUT_TEX = f'../results/ae_gcn_table.tex'

# camera-ready model name (csv) -> new model name (json)
PAIRS = {
    'cellina': 'cellina-ae-cf',
    'cellina-gat': 'cellina-gcn-cf',
}
METRICS = ['pearson', 'direction_match_k', 'edistance_pca_log', 'rmse_lfc']
RAW_METRICS = ['pearson', 'direction_match_k', 'edistance_pca_log', 'mse_lfc']
HOLDOUTS = ['Endothelial', 'Epithelial', 'Fibroblast', 'Myeloid', 'T_cell']

## Load

In [3]:
df_old = pd.read_csv(SUMMARY_CSV)
df_old = df_old[df_old['model_name'].isin(PAIRS.keys())]
df_old = df_old[['sid', 'model_name', 'holdout_celltype'] + RAW_METRICS]
df_old.groupby('model_name').size()

model_name
cellina        30
cellina-gat    30
dtype: int64

In [4]:
corr_dir = os.path.join(DATA_ROOT, f"datasets/{dataset_name}/correlations")

rows = []
for model_name in PAIRS.values():
    for SID in df_old['sid'].unique():
        for ct in HOLDOUTS:
            holdout_celltype = f"{ct}_CRC"
            fp = os.path.join(corr_dir, f"{SID}_{model_name}_{holdout_celltype}.json")
            with open(fp, "r") as f:
                data = json.load(f)
            rows.append({
                "sid": SID,
                "model_name": model_name,
                "holdout_celltype": holdout_celltype,
                **{m: data[m] for m in RAW_METRICS},
            })

df_new = pd.DataFrame(rows)
df_new.groupby('model_name').size()

model_name
cellina-ae-cf     30
cellina-gcn-cf    30
dtype: int64

## Compare

In [5]:
df_all = pd.concat([df_old, df_new], ignore_index=True)
# RMSE_LFC as in make_table.ipynb
df_all['rmse_lfc'] = np.sqrt(df_all['mse_lfc'])
MODEL_ORDER = ['cellina', 'cellina-ae-cf', 'cellina-gat', 'cellina-gcn-cf']
# mean and std pooled over slides and held-out cell types
summary = df_all.groupby('model_name')[METRICS].agg(['mean', 'std']).reindex(MODEL_ORDER)
summary

pearson           direction_match_k           edistance_pca_log            rmse_lfc          
                    mean       std              mean       std              mean       std      mean       std
model_name                                                                                                    
cellina         0.862574  0.173476          0.453333  0.185720          6.997138  1.054692  1.139673  0.560219
cellina-ae-cf   0.887861  0.100161          0.445333  0.165919          6.288954  1.129771  1.069049  0.439752
cellina-gat     0.889207  0.092331          0.418667  0.197077          8.590365  1.895135  1.092881  0.614187
cellina-gcn-cf  0.889168  0.096800          0.434000  0.197023          8.640632  1.343963  1.072642  0.599013

In [6]:
df_all.to_csv(f'../results/ae_gcn_{dataset_name}.csv', index=False)

## LaTeX table

In [7]:
MODEL_RENAME = {
    'cellina': 'Cellina',
    'cellina-ae-cf': 'Cellina-AE',
    'cellina-gat': 'Cellina-GAT',
    'cellina-gcn-cf': 'Cellina-GCN',
}
PRETTY_METRIC = {
    'pearson': r"Pearson $\uparrow$",
    'direction_match_k': r"Signed Precision $\uparrow$",
    'edistance_pca_log': r"E-distance $\downarrow$",
    'rmse_lfc': r"RMSE\textsubscript{LFC} $\downarrow$",
}
BASE_MODELS = ['cellina', 'cellina-ae-cf']
GRAPH_MODELS = ['cellina-gat', 'cellina-gcn-cf']


def fmt(mean, std):
    return f"{mean:.2f} $\\pm$ {std:.2f}"


def row(model):
    return " & ".join([MODEL_RENAME[model]] + [fmt(summary.loc[model, (m, 'mean')], summary.loc[model, (m, 'std')]) for m in METRICS]) + r" \\"


lines = [
    r"\begin{tabular}{l" + "c" * len(METRICS) + "}",
    r"\toprule",
    " & ".join(["Model"] + [PRETTY_METRIC[m] for m in METRICS]) + r" \\",
    r"\midrule",
    *[row(m) for m in BASE_MODELS],
    r"\midrule",
    *[row(m) for m in GRAPH_MODELS],
    r"\bottomrule",
    r"\end{tabular}",
]
latex = "\n".join(lines)
print(latex)

\begin{tabular}{lcccc}
\toprule
Model & Pearson $\uparrow$ & Signed Precision $\uparrow$ & E-distance $\downarrow$ & RMSE\textsubscript{LFC} $\downarrow$ \\
\midrule
Cellina & 0.86 $\pm$ 0.17 & 0.45 $\pm$ 0.19 & 7.00 $\pm$ 1.05 & 1.14 $\pm$ 0.56 \\
Cellina-AE & 0.89 $\pm$ 0.10 & 0.45 $\pm$ 0.17 & 6.29 $\pm$ 1.13 & 1.07 $\pm$ 0.44 \\
\midrule
Cellina-GAT & 0.89 $\pm$ 0.09 & 0.42 $\pm$ 0.20 & 8.59 $\pm$ 1.90 & 1.09 $\pm$ 0.61 \\
Cellina-GCN & 0.89 $\pm$ 0.10 & 0.43 $\pm$ 0.20 & 8.64 $\pm$ 1.34 & 1.07 $\pm$ 0.60 \\
\bottomrule
\end{tabular}


In [8]:
with open(OUT_TEX, 'w') as f:
    f.write(latex + "\n")
print('Saved to', OUT_TEX)

Saved to ../results/ae_gcn_table.tex
